# Rigol DSG836 — Full-Parameter LAN Control & Test

Controls the **Rigol DSG836 RF signal generator** over **LAN (VXI-11)**.
VISA resource string format: `TCPIP0::<IP>::INSTR`

Wraps `instruments.Rigol.DSG836`. **Edit the IP in the config cell before running.**
Every action cell prints a read-back and `dsg.get_error()` so a rejected SCPI
command is caught immediately.

> Requires a VISA backend: NI-VISA, or `pip install pyvisa-py` for pure-Python VXI-11.

In [ ]:
# ── User configuration ────────────────────────────────────────────
DSG836_IP = "192.168.68.40"   # <-- EDIT: Rigol DSG836 LAN IP
# ────────────────────────────────────────────────────

DSG836_VISA = f"TCPIP0::{DSG836_IP}::INSTR"
print("DSG836 resource :", DSG836_VISA)

---
## 0 — List visible VISA resources
Run this first to confirm pyvisa finds the DSG836 on the network.

In [ ]:
import pyvisa

rm = pyvisa.ResourceManager()
print("Visible VISA resources:")
for r in rm.list_resources():
    print(" ", r)

---
## 1 — Imports & sys.path setup

In [ ]:
import sys, os, time

# Make sure the repo root is on the path so `from instruments import ...` works.
REPO_ROOT = os.path.abspath(os.getcwd() if os.path.basename(os.getcwd()) == "AOM" else os.path.join(os.getcwd(), ".."))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
print("Repo root:", REPO_ROOT)

from instruments.Rigol import DSG836
print("Imports OK")

---
## 2 — Connect
`DSG836.__init__` turns modulation, IQ modulation, and RF output **OFF** on connect (safe default).

In [ ]:
dsg = DSG836(DSG836_VISA)
print("DSG836 connected :", dsg.connected)
if dsg.connected:
    print("IDN              :", dsg.idn())
    dsg.clear_status()
    print("Errors           :", dsg.get_error())

---
## 3 — Core RF: frequency, power, output
Frequency range 9 kHz – 3.6 GHz, power −110 – +20 dBm (range-guarded by the class).

In [ ]:
dsg.set_freq(1.0e9)     # 1 GHz
dsg.set_pow(-10.0)      # -10 dBm
dsg.set_output(1)       # RF output ON

print("Freq  (Hz)  :", dsg.get_freq())
print("Power (dBm) :", dsg.get_pow())
print("Output      :", dsg.get_output())
print("Errors      :", dsg.get_error())

---
## 4 — Frequency & level offsets (display-only shifts)

In [ ]:
dsg.set_freq_offset(1.0e6)   # +1 MHz display offset
dsg.set_pow_offset(3.0)      # +3 dB display offset

print("Freq offset (Hz) :", dsg.get_freq_offset())
print("Pow  offset (dB) :", dsg.get_pow_offset())
print("Errors           :", dsg.get_error())

# Reset offsets back to zero
dsg.set_freq_offset(0.0)
dsg.set_pow_offset(0.0)

---
## 5 — Amplitude modulation (AM)

In [ ]:
dsg.set_mod(1)              # master modulation ON
dsg.set_am_source("INT")
dsg.set_am_waveform("SINE")
dsg.set_am_int_freq(1.0e3)  # 1 kHz
dsg.set_am_depth(50.0)      # 50 %
dsg.set_am(1)              # AM ON

print("AM state     :", dsg.get_am())
print("AM depth (%) :", dsg.get_am_depth())
print("AM int freq  :", dsg.get_am_int_freq())
print("Errors       :", dsg.get_error())

dsg.set_am(0)              # AM OFF

---
## 6 — Frequency modulation (FM)

In [ ]:
dsg.set_fm_source("INT")
dsg.set_fm_waveform("SINE")
dsg.set_fm_int_freq(1.0e3)  # 1 kHz
dsg.set_fm_dev(10.0e3)      # 10 kHz deviation
dsg.set_fm(1)              # FM ON

print("FM state    :", dsg.get_fm())
print("FM dev (Hz) :", dsg.get_fm_dev())
print("FM int freq :", dsg.get_fm_int_freq())
print("Errors      :", dsg.get_error())

dsg.set_fm(0)              # FM OFF

---
## 7 — Phase modulation (PM)

In [ ]:
dsg.set_pm_source("INT")
dsg.set_pm_int_freq(1.0e3)  # 1 kHz
dsg.set_pm_dev(1.0)         # 1 rad deviation
dsg.set_pm(1)              # PM ON

print("PM state     :", dsg.get_pm())
print("PM dev (rad) :", dsg.get_pm_dev())
print("PM int freq  :", dsg.get_pm_int_freq())
print("Errors       :", dsg.get_error())

dsg.set_pm(0)              # PM OFF

---
## 8 — Pulse modulation

In [ ]:
dsg.set_pulsemod_src("INT")
dsg.set_pulsemod_mode("SING")
dsg.set_pulse_period(1.0e-3)   # 1 ms period
dsg.set_pulse_width(100.0e-6)  # 100 us width
dsg.set_pulse_polarity("NORM")
dsg.set_pulsemod(1)            # pulse mod ON

print("PULM state    :", dsg.get_pulsemod())
print("Period (s)    :", dsg.get_pulse_period())
print("Width  (s)    :", dsg.get_pulse_width())
print("Errors        :", dsg.get_error())

dsg.set_pulsemod(0)           # pulse mod OFF

---
## 9 — IQ modulation

In [ ]:
dsg.set_iqmod(1)              # IQ mod ON (also enables master mod)
print("IQ state :", dsg.get_iqmod())
print("Errors   :", dsg.get_error())

dsg.set_iqmod(0)             # IQ mod OFF
dsg.set_mod(0)              # master modulation OFF

---
## 10 — Low-frequency (LF) output

In [ ]:
dsg.set_lf_waveform("SINE")
dsg.set_lf_freq(1.0e3)   # 1 kHz
dsg.set_lf_level(0.5)    # 0.5 V
dsg.set_lf(1)           # LF output ON

print("LF state    :", dsg.get_lf())
print("LF freq (Hz):", dsg.get_lf_freq())
print("LF level (V):", dsg.get_lf_level())
print("Errors      :", dsg.get_error())

dsg.set_lf(0)          # LF output OFF

---
## 11 — Frequency sweep
Single linear sweep 1.0 → 1.1 GHz over 11 points.

In [ ]:
dsg.set_sweep_type("FREQ")
dsg.set_sweep_spacing("LIN")
dsg.set_sweep_mode("SING")
dsg.set_sweep_start_freq(1.0e9)
dsg.set_sweep_stop_freq(1.1e9)
dsg.set_sweep_points(11)
dsg.set_sweep_dwell(0.05)     # 50 ms per step
dsg.set_sweep(1)             # enable sweep
dsg.sweep_execute()           # run one sweep

print("Sweep state :", dsg.get_sweep())
print("Errors      :", dsg.get_error())

dsg.set_sweep(0)            # sweep OFF

---
## 12 — Reference clock (internal / external 10 MHz)

In [ ]:
print("Current ref source :", dsg.get_ref_source())

# Example: switch to external 10 MHz reference, then back to internal.
# dsg.set_ref_source("EXT")
# dsg.set_ref_ext_freq(10.0e6)
# print("Ref source now     :", dsg.get_ref_source())

dsg.set_ref_source("INT")
print("Ref source (INT)   :", dsg.get_ref_source())
print("Errors             :", dsg.get_error())

---
## 13 — Full parameter dump (verification snapshot)

In [ ]:
params = {
    "IDN":              dsg.idn(),
    "Frequency (Hz)":   dsg.get_freq(),
    "Power (dBm)":       dsg.get_pow(),
    "Freq offset (Hz)": dsg.get_freq_offset(),
    "Pow offset (dB)":   dsg.get_pow_offset(),
    "RF output":         dsg.get_output(),
    "Master mod":        dsg.get_mod(),
    "AM state":          dsg.get_am(),
    "FM state":          dsg.get_fm(),
    "PM state":          dsg.get_pm(),
    "Pulse mod":         dsg.get_pulsemod(),
    "IQ mod":            dsg.get_iqmod(),
    "LF output":         dsg.get_lf(),
    "Sweep state":       dsg.get_sweep(),
    "Ref source":        dsg.get_ref_source(),
}

print("=== DSG836 parameter snapshot ===")
for k, v in params.items():
    print(f"  {k:<18}: {v}")
print("Errors :", dsg.get_error())

---
## 14 — Teardown
Modulation OFF, RF output OFF, then `*RST` to a known state.

In [ ]:
dsg.set_am(0)
dsg.set_fm(0)
dsg.set_pm(0)
dsg.set_pulsemod(0)
dsg.set_iqmod(0)
dsg.set_lf(0)
dsg.set_mod(0)
dsg.set_output(0)   # RF output OFF
dsg.reset()         # *RST
print("Done. Modulation OFF, RF output OFF, instrument reset.")
print("Errors :", dsg.get_error())

---
## 15 — PulseBlaster: simple channel on/off

Local **SpinCore PulseBlaster** board via the `spinapi64` driver (**USB/PCI, not LAN**).
Uses an inline `PulseBlasterSimple` wrapper built on the repo's `PBESRPro` driver
([instruments/PulseBlaster.py](instruments/PulseBlaster.py)): each channel is held static
with a `CONTINUE`→`BRANCH` loop, and the current mask is tracked so toggling one channel
leaves the others unchanged.

In [ ]:
from instruments.PulseBlaster import PBESRPro


class PulseBlasterSimple(PBESRPro):
    """Minimal static on/off control of individual PulseBlaster TTL channels.

    Holds selected channels high via a CONTINUE->BRANCH static loop (same
    technique as PulseMaster.set_static). Tracks the current output mask so
    toggling one channel leaves the others unchanged. Channels are 0-indexed
    (channel `ch` -> flag bit 1 << ch).
    """

    def __init__(self, clock_MHz=500):
        super().__init__()
        self.pb_init()
        self.pb_core_clock(clock_MHz * self.constants['MHz'])
        self.flags = 0  # current output bitmask

    def _program_static(self, flags):
        self.pb_start_programming(self.PULSE_PROGRAM)
        self.pb_inst_pbonly64(flags, self.inst_set.CONTINUE, 0, 1000)
        self.pb_inst_pbonly64(flags, self.inst_set.BRANCH, 0, 1000)
        self.pb_stop_programming()
        self.pb_start()

    def channel_on(self, ch):
        self.flags |= (1 << ch)
        self._program_static(self.flags)

    def channel_off(self, ch):
        self.flags &= ~(1 << ch)
        self._program_static(self.flags)

    def set_channel(self, ch, state):
        self.channel_on(ch) if state else self.channel_off(ch)

    def get_channel(self, ch):
        return bool(self.flags & (1 << ch))

    def all_off(self):
        self.flags = 0
        self._program_static(0)
        self.pb_stop()


print("PulseBlasterSimple defined")

In [ ]:
pb = PulseBlasterSimple(clock_MHz=500)   # inits board + sets 500 MHz core clock
print("Boards detected :", pb.pb_count_boards())
print("Driver version  :", pb.pb_get_version())

In [ ]:
PB_CHANNEL = 2   # <-- EDIT: PulseBlaster output channel (0-indexed)

pb.channel_on(PB_CHANNEL)
print(f"Channel {PB_CHANNEL} ON  -> state:", pb.get_channel(PB_CHANNEL))

# When ready to turn it off, run:
# pb.channel_off(PB_CHANNEL)
# print(f"Channel {PB_CHANNEL} OFF -> state:", pb.get_channel(PB_CHANNEL))

In [ ]:
pb.all_off()     # all channels OFF, board stopped
pb.pb_close()    # release the board
print("PulseBlaster: all channels OFF, board closed.")